# 03 — Architecture + Pretraining (Wikitext-103)

Pretrains the GPT-style transformer defined in [`model.py`](model.py) —
`Head`, `MultiHeadAttention`, `FeedForward`, `Block`, `GPTLanguageModel` —
on Wikitext-103 tokens, using plain next-token prediction (cross-entropy
loss) and nothing else: no pretrained weights, no pretrained tokenizer, no
Hugging Face model classes.

**Architecture** (see `model.py` for the full implementation and comments):

| | |
|---|---|
| Parameters | **33.5M** |
| Embedding dim (`n_embd`) | 512 |
| Attention heads (`n_head`) | 8 |
| Transformer blocks (`n_layer`) | 8 |
| Context length (`block_size`) | 256 |
| Dropout | 0.2 |
| Vocabulary | 8,000 (from `02_tokenizer.ipynb`) |

**Training:** 37.5M tokens (a 150M-character sample of Wikitext-103,
encoded with the BPE tokenizer), 90/10 train/val split, AdamW,
lr 3e-4, batch size 16, trained to step 14,999 —
**final train loss 3.7288, val loss 3.8494.**

Checkpoints aren't committed to this repo (419MB, over GitHub's file size
limit) — re-run this notebook to regenerate `model_step14999.pt`. Expect
roughly 15,000 steps to take a few hours on a single Colab T4 GPU.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tokenizers torch
print("Ready.")

Mounted at /content/drive
Ready.

In [1]:
import torch
from tokenizers import Tokenizer
import numpy as np

# --- Load tokenizer ---
tokenizer_path = '/content/drive/MyDrive/sec_chatbot/tokenizer/tokenizer.json'
tokenizer = Tokenizer.from_file(tokenizer_path)
vocab_size = tokenizer.get_vocab_size()
print(f"Tokenizer loaded. Vocabulary size: {vocab_size}")

# --- Read and encode Wikitext-103 in chunks (keeps memory bounded) ---
wiki_path = '/content/drive/MyDrive/sec_chatbot/data/wikitext103_train.txt'
target_chars = 150_000_000
chunk_size = 5_000_000

all_ids = []
chars_read = 0

print("Encoding in chunks (this takes a few minutes)...")
with open(wiki_path, 'r', encoding='utf-8') as f:
    while chars_read < target_chars:
        chunk = f.read(chunk_size)
        if not chunk:
            break
        ids = tokenizer.encode(chunk).ids
        all_ids.append(np.array(ids, dtype=np.uint16))  # uint16: far less memory than a Python list
        chars_read += len(chunk)
        print(f"  {chars_read:,} characters encoded")

data_np = np.concatenate(all_ids)
del all_ids
data = torch.from_numpy(data_np.astype(np.int64))
del data_np

print(f"\nTotal tokens: {len(data):,}")

# --- Split ---
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]
print(f"Training tokens: {len(train_data):,}")
print(f"Validation tokens: {len(val_data):,}")

Tokenizer loaded. Vocabulary size: 8000
Encoding in chunks (this takes a few minutes)...
  5,000,000 characters encoded
  ...
  150,000,000 characters encoded

Total tokens: 37,539,260
Training tokens: 33,785,334
Validation tokens: 3,753,926

In [1]:
import sys
sys.path.append('/content/drive/MyDrive/sec_chatbot/scripts')  # wherever model.py lives
from model import block_size  # single source of truth for context length

batch_size = 16

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

train_data = train_data.to(device)
val_data = val_data.to(device)

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print(f"Input batch shape: {xb.shape}")
print(f"Target batch shape: {yb.shape}")

Using device: cuda
Input batch shape: torch.Size([16, 256])
Target batch shape: torch.Size([16, 256])

In [1]:
from model import GPTLanguageModel

model = GPTLanguageModel(vocab_size).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model created with {n_params/1e6:.1f} million parameters")

Model created with 33.5 million parameters

In [1]:
import os
import glob

# --- Training settings ---
max_iters = 15000
eval_interval = 500
eval_iters = 200
learning_rate = 3e-4

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

ckpt_dir = '/content/drive/MyDrive/sec_chatbot/checkpoints'
os.makedirs(ckpt_dir, exist_ok=True)

# --- Resume from the latest checkpoint if one exists (Colab sessions time out) ---
start_iter = 0
checkpoints = glob.glob(os.path.join(ckpt_dir, 'model_step*.pt'))
if checkpoints:
    latest = max(checkpoints, key=lambda p: int(p.split('step')[1].split('.')[0]))
    print(f"Found checkpoint: {latest}\nResuming from it...")
    ckpt = torch.load(latest)
    model.load_state_dict(ckpt['model_state'])
    optimizer.load_state_dict(ckpt['optimizer_state'])
    start_iter = ckpt['step'] + 1
    print(f"Resumed. Continuing from step {start_iter}")
else:
    print("No checkpoint found. Starting fresh from step 0.")

# --- Training loop ---
print("\nStarting training...\n")
for iter in range(start_iter, max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"Step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
        ckpt_path = os.path.join(ckpt_dir, f'model_step{iter}.pt')
        torch.save({
            'step': iter,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'train_loss': losses['train'].item(),
            'val_loss': losses['val'].item(),
        }, ckpt_path)

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("\nTraining complete.")

Found checkpoint: /content/drive/MyDrive/sec_chatbot/checkpoints/model_step12500.pt
Resuming from it...
Resumed. Continuing from step 12501

Starting training...

Step 13000: train loss 3.7899, val loss 3.9181
Step 13500: train loss 3.7797, val loss 3.8949
Step 14000: train loss 3.7483, val loss 3.8888
Step 14500: train loss 3.7341, val loss 3.8624
Step 14999: train loss 3.7288, val loss 3.8494

Training complete.

In [1]:
from torch.nn import functional as F

@torch.no_grad()
def generate(model, max_new_tokens=200):
    bos_id = tokenizer.token_to_id("<BOS>")
    idx = torch.tensor([[bos_id]], dtype=torch.long, device=device)

    for _ in range(max_new_tokens):
        idx_cond = idx[:, -block_size:]  # model can't see further back than block_size
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :]
        probs = F.softmax(logits, dim=-1)
        idx_next = torch.multinomial(probs, num_samples=1)
        idx = torch.cat((idx, idx_next), dim=1)

    return tokenizer.decode(idx[0].tolist())

print("--- GENERATED TEXT (pretrained, before any SEC fine-tuning) ---\n")
print(generate(model, max_new_tokens=200))

--- GENERATED TEXT (pretrained, before any SEC fine-tuning) ---

Leg ends to The C lo ose . The premi ere of the Str ate Cor mon venue was not the legal av iation of the n inet een million years , with additional p up ils reflect ing sched ule to a survey of the corpor ation , as covered between the 12 @-@ quarters @-@ park stations over value of their p up pet . The rock track ing makes them achie ving a spring at the west side to London : that remained for nine years . The Blue park was scheduled to travel without the build ing tour , as teams hand led into the Football Association of Wales . One day comes were program ming from place S ports Day ; however , the introduction of the cr amp ed tracks are closed to that of centre goods and game collect ions and that would have been in or managed , and instead require s now w icket or w ag red med all to work using the stadium after each summer catch er , South London that the stadium would only have stopped in the town even though they 

**Reading the output above:** at 33.5M parameters and ~37.5M pretraining
tokens, the model has clearly learned subword structure (real English
morphemes, correct-ish punctuation spacing from the BPE tokenizer) and the
rough shape of Wikipedia-style prose (place names, dates, article-like
phrasing) — but not long-range coherence. That's expected at this scale;
the point of pretraining here is a reasonable general-English starting
point for `04_finetune.ipynb` to specialize, not a fluent generator on its
own.